# 03. AI Search 쿼리 및 RAG 테스트

AI Search 인덱스에서 법령 데이터를 검색하고, GPT-5.4 기반 RAG 질의응답을 테스트합니다.

⚠️ **실행 위치 필수 조건**
- 이 노트북(03)은 **내부망(VNet/Private Endpoint)에 연결된 Remote VM**에서 실행해야 합니다.
- 로컬 PC/외부망 환경에서는 Search/OpenAI Private Endpoint DNS 해석 또는 네트워크 경로 문제로 실패할 수 있습니다.

## 검색 유형
| 유형 | 방식 |
|------|------|
| 키워드 검색 | BM25 전통적 텍스트 매칭 |
| 벡터 검색 | text-embedding-3-large (3072D) 의미 기반 |
| 하이브리드 검색 | 키워드 + 벡터 RRF 결합 |
| 시맨틱 재랭킹 | Azure AI Search Semantic Ranker |
| RAG | 하이브리드 검색 + **GPT-5.4** 응답 생성 |

## 인덱스 구조 (AI Search Skills 자동 생성)
```
Blob(raw-documents) → [Split 2000자/200자OV] → [Embedding 3072D] → Index
```

## 1. 환경 설정

In [ ]:
import os
import sys
import requests as req
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

SEARCH_ENDPOINT  = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT  = os.environ['AZURE_OPENAI_ENDPOINT']
INDEX_NAME       = os.environ.get('AZURE_SEARCH_INDEX_NAME', 'law-documents-index')
EMBEDDING_DEPLOY = os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')
GPT54_DEPLOY     = os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4')

print(f'Search Endpoint : {SEARCH_ENDPOINT}')
print(f'OpenAI Endpoint : {OPENAI_ENDPOINT}')
print(f'Index           : {INDEX_NAME}')
print(f'GPT-5.4 배포명  : {GPT54_DEPLOY}')

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery, QueryType, QueryCaptionType, QueryAnswerType
from openai import AzureOpenAI

credential = DefaultAzureCredential()

# AI Search 클라이언트 (Managed Identity)
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential
)

# OpenAI 클라이언트 (Managed Identity)
token_provider = get_bearer_token_provider(
    credential, 'https://cognitiveservices.azure.com/.default'
)
openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version='2025-01-01-preview',
)

# 인덱스 문서 수 확인
results = search_client.search('*', top=0, include_total_count=True)
print(f'인덱스 문서 수: {results.get_count()}')

## 인덱스 스키마 설계 — 법률 4개 인덱스

법률 데이터 특성에 맞게 인덱스별로 스키마를 분리 설계합니다.  
모든 인덱스는 **하이브리드 검색(키워드 + 벡터) + Semantic Ranker** 구조를 공유합니다.

---

### 공통 설계 원칙

| 구분 | 클래스 | 설정 | 용도 |
|------|--------|------|------|
| **Key 필드** | `SimpleField` | `key=True, filterable=True` | 문서 고유 ID |
| **날짜/번호 메타** | `SimpleField` | `filterable=True, sortable=True` | 날짜 범위 필터, 정렬 |
| **열거형 메타** | `SearchableField` | `filterable=True, facetable=True` | 심급·유형 패싯 UI |
| **다중값 메타** | `SearchableField` | `type=Collection(String), filterable=True` | 관련법령·주제어 정확 필터 |
| **본문 텍스트** | `SearchableField` | `analyzer_name="ko.microsoft"` | 한국어 키워드 검색 |
| **벡터 임베딩** | `SearchField` | `searchable=True, hidden=True` | 의미 기반 벡터 검색 |

#### 주요 설계 결정

**`relatedLaws` / `keywords` → `Collection(String)` 타입**  
단일 String으로 "민법,형법"을 저장하면 "민법" 단독 필터가 불가능합니다.  
Collection 타입으로 변경하면 OData 람다 표현식으로 정확히 필터링할 수 있습니다.
```
# 특정 법령 포함 문서만 필터
filter="relatedLaws/any(law: law eq '민법')"

# 여러 법령 중 하나라도 포함
filter="relatedLaws/any(law: law eq '민법' or law eq '형법')"

# 데이터 인덱싱 시 배열로 제공
{ "relatedLaws": ["민법", "형법", "행정소송법"] }
```

**`summaryEmbedding` → `hidden=True`**  
검색 응답에 3072개 float가 포함되면 네트워크 비용이 크게 증가합니다.  
`hidden=True`로 벡터 필드를 결과에서 제외하고, 검색(scoring)에만 활용합니다.

**`caseNumber` / `docNumber` → `SearchableField`**  
"2024다12345" 형태의 사건번호를 키워드 검색으로 찾을 수 있도록 searchable을 추가합니다.

**Semantic Search 설정 (인덱스별)**  
Azure AI Search Semantic Ranker가 L2R 재랭킹을 수행하도록 각 인덱스에 semantic configuration을 등록합니다.

### 인덱스별 필드 스키마

#### 1. `prec-court-index` — 판례 (대법원·고등법원·지방법원)

| 필드명 | 타입 | Key | Srch | Filt | Sort | Facet | 비고 |
|--------|------|:---:|:----:|:----:|:----:|:-----:|------|
| `id` | String | ✅ | | ✅ | | | 고유 ID |
| `courtName` | String | | ✅ | ✅ | | ✅ | 법원명 |
| `caseNumber` | String | | ✅ | ✅ | ✅ | | 사건번호 (텍스트 검색 지원) |
| `judgmentDate` | DateTimeOffset | | | ✅ | ✅ | | 선고일자 |
| `courtLevel` | String | | ✅ | ✅ | | ✅ | 심급 (1심/2심/3심) |
| `caseType` | String | | ✅ | ✅ | | ✅ | 사건종류 (민사/형사/행정) |
| `status` | String | | ✅ | ✅ | | ✅ | 진행상태 (확정/계류) |
| `relatedLaws` | **Collection(String)** | | ✅ | ✅ | | | 관련법령 배열 |
| `keywords` | **Collection(String)** | | ✅ | ✅ | | | 주제어 배열 |
| `registrationDate` | DateTimeOffset | | | ✅ | ✅ | | 등록일자 |
| `caseName` | String | | ✅ | | | | 사건명 (ko.microsoft) |
| `holdings` | String | | ✅ | | | | 판시사항 (ko.microsoft) |
| `summary` | String | | ✅ | | | | 판결요지 (ko.microsoft) |
| `fullText` | String | | ✅ | | | | 판결 전문 (ko.microsoft) |
| `summaryEmbedding` | Collection(Single) | | ✅ | | | | 벡터 3072D, **hidden** |

**Semantic config** `prec-semantic`: title=`caseName` / content=`holdings,summary` / keywords=`keywords,relatedLaws`

---

#### 2. `const-court-index` — 헌법재판소 결정례

| 필드명 | 타입 | Key | Srch | Filt | Sort | Facet | 비고 |
|--------|------|:---:|:----:|:----:|:----:|:-----:|------|
| `id` | String | ✅ | | ✅ | | | 고유 ID |
| `caseNumber` | String | | ✅ | ✅ | ✅ | | 사건번호 (텍스트 검색 지원) |
| `decisionDate` | DateTimeOffset | | | ✅ | ✅ | | 선고일자 |
| `decisionType` | String | | ✅ | ✅ | | ✅ | 결정유형 (합헌/위헌/헌법불합치/각하) |
| `relatedLaws` | **Collection(String)** | | ✅ | ✅ | | | 관련법령 배열 |
| `keywords` | **Collection(String)** | | ✅ | ✅ | | | 주제어 배열 |
| `fiscalYear` | String | | | ✅ | ✅ | | 귀속년도 |
| `registrationDate` | DateTimeOffset | | | ✅ | ✅ | | 등록일자 |
| `caseName` | String | | ✅ | | | | 사건명 (ko.microsoft) |
| `holdings` | String | | ✅ | | | | 판시사항 (ko.microsoft) |
| `summary` | String | | ✅ | | | | 결정요지 (ko.microsoft) |
| `fullText` | String | | ✅ | | | | 전문 (ko.microsoft) |
| `summaryEmbedding` | Collection(Single) | | ✅ | | | | 벡터 3072D, **hidden** |

**Semantic config** `const-semantic`: title=`caseName` / content=`holdings,summary` / keywords=`keywords,relatedLaws`

---

#### 3. `legis-interp-index` — 법제처 해석례

| 필드명 | 타입 | Key | Srch | Filt | Sort | Facet | 비고 |
|--------|------|:---:|:----:|:----:|:----:|:-----:|------|
| `id` | String | ✅ | | ✅ | | | 고유 ID |
| `docNumber` | String | | ✅ | ✅ | ✅ | | 문서번호 (텍스트 검색 지원) |
| `replyDate` | DateTimeOffset | | | ✅ | ✅ | | 회신일자 |
| `interpType` | String | | ✅ | ✅ | | ✅ | 안건종류 (법령해석/법제업무) |
| `relatedLaws` | **Collection(String)** | | ✅ | ✅ | | | 관련법령 배열 |
| `keywords` | **Collection(String)** | | ✅ | ✅ | | | 주제어 배열 |
| `registrationDate` | DateTimeOffset | | | ✅ | ✅ | | 등록일자 |
| `title` | String | | ✅ | | | | 안건명 (ko.microsoft) |
| `querySummary` | String | | ✅ | | | | 질의요지 (ko.microsoft) |
| `reply` | String | | ✅ | | | | 회답 (ko.microsoft) |
| `reason` | String | | ✅ | | | | 이유 (ko.microsoft) |
| `summaryEmbedding` | Collection(Single) | | ✅ | | | | 벡터 3072D, **hidden** |

**Semantic config** `interp-semantic`: title=`title` / content=`querySummary,reply` / keywords=`keywords,relatedLaws`

---

#### 4. `admin-appeal-index` — 행정심판 재결례

| 필드명 | 타입 | Key | Srch | Filt | Sort | Facet | 비고 |
|--------|------|:---:|:----:|:----:|:----:|:-----:|------|
| `id` | String | ✅ | | ✅ | | | 고유 ID |
| `caseNumber` | String | | ✅ | ✅ | ✅ | | 사건번호 (텍스트 검색 지원) |
| `decisionDate` | DateTimeOffset | | | ✅ | ✅ | | 재결일자 |
| `decisionType` | String | | ✅ | ✅ | | ✅ | 재결유형 (인용/기각/각하) |
| `relatedLaws` | **Collection(String)** | | ✅ | ✅ | | | 관련법령 배열 |
| `keywords` | **Collection(String)** | | ✅ | ✅ | | | 주제어 배열 |
| `committee` | String | | ✅ | ✅ | | ✅ | 심판위원회 |
| `registrationDate` | DateTimeOffset | | | ✅ | ✅ | | 등록일자 |
| `caseName` | String | | ✅ | | | | 사건명 (ko.microsoft) |
| `order` | String | | ✅ | | | | 주문 (ko.microsoft) |
| `reasonSummary` | String | | ✅ | | | | 이유요약 (ko.microsoft) |
| `fullText` | String | | ✅ | | | | 전문 (ko.microsoft) |
| `summaryEmbedding` | Collection(Single) | | ✅ | | | | 벡터 3072D, **hidden** |

**Semantic config** `admin-semantic`: title=`caseName` / content=`order,reasonSummary` / keywords=`keywords,relatedLaws`

---

### 벡터 검색 설정 (전 인덱스 공통)

```
Algorithm : HNSW (Hierarchical Navigable Small World)
Metric    : Cosine Similarity
Dimensions: 3072  (text-embedding-3-large)
m         : 4     (이웃 연결 수)
ef_construction: 400  (인덱싱 시 탐색 범위)
ef_search : 500   (쿼리 시 탐색 범위)
```

In [ ]:
import sys
sys.path.insert(0, '..')
from src.search.legal_indexes import (
    LegalIndexManager, PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX
)

# 인덱스 스키마 요약 출력
INDEX_META = {
    PREC_INDEX: {
        "name": "판례",
        "embedding_fields": ["holdings", "summary"],
        "semantic_config": "prec-semantic",
        "facet_fields": ["courtName", "courtLevel", "caseType", "status"],
    },
    CONST_INDEX: {
        "name": "헌재결정례",
        "embedding_fields": ["holdings", "summary"],
        "semantic_config": "const-semantic",
        "facet_fields": ["decisionType"],
    },
    INTERP_INDEX: {
        "name": "법제처 해석례",
        "embedding_fields": ["querySummary", "reply"],
        "semantic_config": "interp-semantic",
        "facet_fields": ["interpType"],
    },
    ADMIN_INDEX: {
        "name": "행정심판 재결례",
        "embedding_fields": ["order", "reasonSummary"],
        "semantic_config": "admin-semantic",
        "facet_fields": ["decisionType", "committee"],
    },
}

print(f"{'인덱스명':<30} {'한국어명':<15} {'임베딩 대상':<30} {'Semantic Config':<20} {'Facet 필드'}")
print("-" * 110)
for idx, meta in INDEX_META.items():
    print(
        f"{idx:<30} {meta['name']:<15} "
        f"{', '.join(meta['embedding_fields']):<30} "
        f"{meta['semantic_config']:<20} "
        f"{', '.join(meta['facet_fields'])}"
    )

### Collection(String) 필터 사용 예시

`relatedLaws`와 `keywords`는 배열 타입이므로 OData 람다 표현식으로 필터링합니다.

In [ ]:
# Collection(String) OData 필터 예시
filter_examples = {
    "특정 법령 포함": "relatedLaws/any(law: law eq '민법')",
    "여러 법령 중 하나 포함": "relatedLaws/any(law: law eq '민법' or law eq '형법')",
    "특정 키워드 포함": "keywords/any(kw: kw eq '손해배상')",
    "법령 + 날짜 범위 복합": "relatedLaws/any(law: law eq '민법') and judgmentDate ge 2020-01-01T00:00:00Z",
    "위헌 결정만 + 특정 법령": "decisionType eq '위헌' and relatedLaws/any(law: law eq '헌법')",
}

print("=== Collection(String) OData 필터 표현식 ===\n")
for desc, expr in filter_examples.items():
    print(f"[{desc}]")
    print(f"  filter=\"{expr}\"\n")

# Semantic Ranker 검색 예시 (실제 연결 시 실행)
print("=== Semantic Ranker 검색 예시 ===\n")
semantic_search_example = '''
from azure.search.documents.models import VectorizedQuery, QueryType, QueryCaptionType

results = search_client.search(
    search_text="손해배상 책임 범위",
    vector_queries=[VectorizedQuery(vector=embedding, k=50, fields="summaryEmbedding")],
    query_type=QueryType.SEMANTIC,
    semantic_configuration_name="prec-semantic",    # [Fix 3] Semantic Ranker
    query_caption=QueryCaptionType.EXTRACTIVE,
    filter="relatedLaws/any(law: law eq '민법')",   # [Fix 1] Collection 필터
    top=5,
    # summaryEmbedding은 hidden=True라 자동 제외됨  # [Fix 2]
)
for r in results:
    score = r.get("@search.reranker_score", r["@search.score"])
    print(f"[{score:.4f}] {r['caseName']}")
    for caption in r.get("@search.captions", []):
        print(f"  Caption: {caption.text}")
'''
print(semantic_search_example)

## 2. 키워드 검색

In [ ]:
query = '개인정보 보호'

results = search_client.search(
    search_text=query,
    top=5,
    select=['content', 'metadata_storage_name', 'metadata_storage_path']
)

print(f"검색어: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"--- {i} [{doc['@search.score']:.4f}] {doc.get('metadata_storage_name', 'N/A')} ---")
    print(doc.get('content', '')[:200])
    print()

## 3. 하이브리드 검색 + 시맨틱 재랭킹

In [ ]:
query = '법령 개정 절차와 공포 방법은 어떻게 되나요?'

results = search_client.search(
    search_text=query,
    vector_queries=[
        VectorizableTextQuery(text=query, k_nearest_neighbors=50, fields='content_vector')
    ],
    query_type=QueryType.SEMANTIC,
    semantic_configuration_name='law-semantic-config',
    query_caption=QueryCaptionType.EXTRACTIVE,
    query_answer=QueryAnswerType.EXTRACTIVE,
    top=5,
    select=['content', 'metadata_storage_name']
)

print(f"검색어: '{query}'\n")
for i, doc in enumerate(results, 1):
    score = doc.get('@search.reranker_score', doc.get('@search.score', 0))
    print(f"--- {i} [reranker: {score:.4f}] {doc.get('metadata_storage_name', 'N/A')} ---")
    captions = doc.get('@search.captions', [])
    if captions:
        print(f'  Caption: {captions[0].text}')
    else:
        print(f"  {doc.get('content', '')[:200]}")
    print()

## 4. RAG — GPT-5.4 기반 질의응답

AI Search 하이브리드 검색 결과를 컨텍스트로 GPT-5.4에 전달하여 정확한 법령 기반 답변을 생성합니다.

In [ ]:
def rag_query(question: str, top_k: int = 5, model: str = GPT54_DEPLOY) -> str:
    """하이브리드 검색 + GPT-5.4 RAG 질의응답"""
    # 1. 하이브리드 + 시맨틱 검색
    search_results = list(search_client.search(
        search_text=question,
        vector_queries=[
            VectorizableTextQuery(text=question, k_nearest_neighbors=50, fields='content_vector')
        ],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name='law-semantic-config',
        top=top_k,
        select=['content', 'metadata_storage_name']
    ))

    # 2. 컨텍스트 구성
    context = '\n\n'.join(
        f"[출처: {r.get('metadata_storage_name', '알 수 없음')}]\n{r.get('content', '')}"
        for r in search_results
    )

    # 3. GPT-5.4 응답 생성
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': (
                    '당신은 한국 법령 전문가입니다. '
                    '주어진 검색 결과를 바탕으로 사용자의 질문에 정확하고 상세하게 답변하세요. '
                    '답변의 근거가 되는 법령 출처를 반드시 명시하세요. '
                    '검색 결과에 없는 내용은 검색된 자료에 해당 내용이 없습니다라고 답변하세요.'
                ),
            },
            {
                'role': 'user',
                'content': f'## 검색된 법령 자료\n\n{context}\n\n## 질문\n\n{question}',
            },
        ],
        max_tokens=2000,
        temperature=0.3,
    )
    return response.choices[0].message.content

print(f'RAG 함수 준비 완료 (기본 모델: {GPT54_DEPLOY})')

In [ ]:
from IPython.display import Markdown, display

question = '개인정보 보호법에서 정보주체의 권리는 무엇인가요?'
answer = rag_query(question)

display(Markdown(f'### 질문\n{question}\n\n### GPT-5.4 답변\n{answer}'))

In [ ]:
questions = [
    '인공지능 산업 육성 및 신뢰 확보에 관한 법률의 주요 내용은?',
    '클라우드컴퓨팅 서비스 사업자의 의무사항을 알려주세요.',
    '전자상거래에서 소비자 보호 규정은 어떻게 되나요?',
]

for q in questions:
    answer = rag_query(q)
    display(Markdown(f'---\n**Q**: {q}\n\n**A (GPT-5.4)**: {answer[:600]}...'))

## 6. 인덱서 상태 및 인덱스 통계

In [ ]:
token = credential.get_token('https://search.azure.com/.default').token
headers = {'Authorization': f'Bearer {token}'}

# 인덱스 통계
r = req.get(f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/stats?api-version=2024-07-01', headers=headers)
stats = r.json()
print(f"문서 수: {stats.get('documentCount', 0):,}")
print(f"Storage: {stats.get('storageSize', 0) / 1024:.1f} KB")

# 인덱서 마지막 실행 상태
r2 = req.get(f'{SEARCH_ENDPOINT}/indexers/law-blob-indexer/status?api-version=2024-07-01', headers=headers)
last = r2.json().get('lastResult', {})
print(f"\n인덱서 마지막 실행")
print(f"  상태: {last.get('status', 'N/A')}")
print(f"  처리: {last.get('itemsProcessed', 0)}건")
print(f"  실패: {last.get('itemsFailed', 0)}건")
print(f"  완료: {last.get('endTime', 'N/A')}")

---
## Lab 완료!

이 Hands-on Lab을 통해 다음을 학습했습니다:

1. **인프라 배포**: Bicep으로 Sweden Central Private Network 전체 인프라 자동 배포
2. **크롤링 자동화**: Logic Apps + Azure Function (EP1, VNet Integration)으로 Azure에서 실행
3. **AI Search Skills 파이프라인**: Document Intelligence → Split → Embedding 자동 인덱싱
4. **하이브리드 검색 + RAG**: GPT-5.4 기반 법령 질의응답

다음 단계: [04-legal-multi-index.ipynb](04-legal-multi-index.ipynb) — 법령 다중 인덱스 구성